# Laboratorium 4: od przewidywania do modelu liniowego

Jak przewidywać sprzedaż na podstawie budżetu reklamowego?
Zaczniemy od jednej wspólnej predykcji, zbudujemy własną prostą, a potem porównamy ją z prostą dopasowaną przez bibliotekę.

[Karta laboratorium](https://rodakt.github.io/MAD/laby/04-regresja-liniowa.html)

Zapisz kopię notebooka na swoim Dysku (*Plik → Zapisz kopię na Dysku*).
Kod wpisuj w pustych komórkach. Podpowiedzi otwieraj wtedy, gdy ich potrzebujesz.
Przed obliczeniem spróbuj przewidzieć wynik, a po obliczeniu zapisz krótką interpretację
w komórce „Twoja odpowiedź” (dwuklik pozwala ją edytować).

Na części 1--6 mamy około 75 minut, na wnioski 5 minut. Pozostałe 10 minut to rezerwa na pytania.
Rozszerzenie na końcu jest opcjonalne.

## Przygotowanie

Uruchom poniższą komórkę. Wczytuje dane *Advertising* używane w rozdziale 6 książki
i dzieli je na 150 rynków uczących (`data_tr`) oraz 50 testowych (`data_te`).
Na razie pracujemy tylko na `data_tr`; `data_te` otworzymy w części 5.
Ten sam podział zachowujemy przez całe zajęcia.

Każdy wiersz opisuje jeden rynek. Budżety reklamy `TV`, `radio`, `newspaper` są podane
w tysiącach USD, a sprzedaż `sales` w tysiącach sztuk.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split

advertising = pd.read_csv("https://rodakt.github.io/MAD/laby/dane/Advertising.csv", index_col=0)
data_tr, data_te = train_test_split(advertising, test_size=0.25, random_state=2026)

## 1. Co jest jedną obserwacją? (8 min)

### 1.1. Obejrzyj dane

Wyświetl pięć pierwszych wierszy próby uczącej.
Wybierz jeden rynek: jakie ma budżety i jaką sprzedaż? Podaj wartości w dolarach i sztukach.

<details>
<summary>Podpowiedź: pierwsze wiersze</summary>

`data_tr.head()` wyświetla początek tabeli.

</details>

**Twoja odpowiedź:**

### 1.2. Zobacz zależność

Narysuj sprzedaż względem budżetu TV na rynkach uczących. Opisz obie osie wraz z jednostkami.
Czy prosta wydaje się rozsądnym przybliżeniem? Czy rynki o podobnym budżecie zawsze mają podobną sprzedaż?

<details>
<summary>Podpowiedź 1: wybór kolumny</summary>

Kolumnę tabeli wybierasz przez `data_tr["TV"]`.

</details>

<details>
<summary>Podpowiedź 2: wykres</summary>

`plt.scatter(x, Y)` rysuje punkty o współrzędnych z dwóch podanych kolumn.

</details>

<details>
<summary>Podpowiedź 3: opisy osi</summary>

`plt.xlabel("budżet TV (tys. USD)")` opisuje oś poziomą. Oś pionową opisuje `plt.ylabel(...)`. Na końcu użyj `plt.show()`.

</details>

**Twoja odpowiedź:**

## 2. Pierwszy model: każdemu tyle samo (10 min)

### 2.1. Jedna predykcja i jeden rynek

Załóż, że na każdym rynku przewidujesz sprzedaż 10 tys. sztuk.
Zapisz tę wartość jako `prediction`. Odczytaj sprzedaż pierwszego rynku w `data_tr`
i odejmij od niej predykcję. Co oznacza znak otrzymanej różnicy?

<details>
<summary>Podpowiedź: pierwszy element</summary>

`data_tr["sales"].iloc[0]` wybiera sprzedaż pierwszego rynku w tabeli. Indeks przy wierszu może mieć inny numer.

</details>

**Twoja odpowiedź:**

### 2.2. Pomyłki na wszystkich rynkach

Oblicz różnicę „obserwacja minus predykcja” dla każdego rynku uczącego i zapisz ją jako `errors`.
Wyświetl kilka różnic, a potem ich średnią.
Czy mała średnia różnic gwarantuje małe pomyłki? Co się stanie dla pomyłek +5 i −5?

<details>
<summary>Podpowiedź 1: wszystkie obserwacje</summary>

Od kolumny możesz odjąć liczbę. Odejmowanie zostanie wykonane dla każdego elementu.

</details>

<details>
<summary>Podpowiedź 2: średnia</summary>

Średnią elementów oblicza metoda `.mean()`.

</details>

**Twoja odpowiedź:**

### 2.3. Jedna liczba mierząca wielkość pomyłek

Podnieś każdą różnicę do kwadratu, a następnie oblicz średnią tych kwadratów.
Zapisz wynik jako `mse_constant`. To błąd średniokwadratowy (MSE).
Dlaczego przeciwne znaki pomyłek już się nie znoszą? Czy mniejsza wartość tej miary jest lepsza?

<details>
<summary>Podpowiedź: kwadraty elementów</summary>

`errors**2` podnosi każdy element do kwadratu.

</details>

**Twoja odpowiedź:**

### 2.4. Zaproponuj lepszą stałą

Wybierz inną wspólną predykcję niż 10. Zapisz ją jako `other_prediction`
i oblicz jej MSE na tych samych rynkach. Czy wynik poprawił się?

**Twoja odpowiedź:**

### 2.5. Średnia jako punkt odniesienia

Oblicz średnią sprzedaż na rynkach uczących i zapisz ją jako `mean_tr`.
Przewiduj tę samą wartość na każdym rynku. Oblicz MSE takiego modelu jako `mse_mean`.
Porównaj wynik z dwiema poprzednimi stałymi. Zachowaj `mean_tr`: przyda się na nowych danych.

Średnia z próby minimalizuje MSE wśród wspólnych stałych predykcji; tutaj sprawdzamy to na swoich przykładach.

<details>
<summary>Podpowiedź: średnia sprzedaż</summary>

`data_tr["sales"].mean()` oblicza średnią w próbie uczącej.

</details>

**Twoja odpowiedź:**

## 3. Własna prosta: co robią parametry? (15 min)

Teraz predykcja ma zależeć od budżetu TV:
$$\widehat y = a + b x.$$
Na początek przyjmij $a=5$, $b=0{,}04$. Budżet $x$ i predykcja nadal są w tysiącach.

**Najpierw bez kodu:** ile przewidzisz dla budżetu 100 tys. USD?
O ile zmieni się predykcja przy budżecie większym o 10 tys. USD?

**Twoja odpowiedź:**

### 3.1. Oblicz predykcje

Zapisz `a = 5` i `b = 0.04`. Oblicz predykcję dla każdego rynku uczącego i zapisz wyniki jako `Y_hat`.
Wyświetl kilka predykcji. Czy wszystkie są takie same?

<details>
<summary>Podpowiedź: od liczby do kolumny</summary>

W równaniu zastąp pojedynczy budżet całą kolumną `data_tr["TV"]`.

</details>

**Twoja odpowiedź:**

### 3.2. Narysuj swoją prostą

Narysuj ponownie punkty z części 1. Dodaj swoją prostą.
Do narysowania odcinka wystarczą dwa budżety: 0 i 300 tys. USD.
Zapisz je jako `grid` i oblicz dla nich predykcje.

<details>
<summary>Podpowiedź 1: dwa budżety</summary>

`np.array([0, 300])` tworzy tablicę dwóch liczb.

</details>

<details>
<summary>Podpowiedź 2: odcinek</summary>

`plt.plot(grid, a + b * grid)` łączy dwa punkty. Wpisz tę linię między rysowaniem punktów a `plt.show()`.

</details>

### 3.3. Zmieniaj po jednym parametrze

Przed uruchomieniem zapisz, co stanie się z prostą, gdy zwiększysz samo `a`.
Sprawdź przewidywanie na wykresie.
Następnie wróć do `a = 5`, zwiększ samo `b` i powtórz eksperyment.
Możesz skopiować swój kod wykresu do poniższej komórki.

W odpowiedzi wyjaśnij różnicę między rolą `a` i rolą `b`.

**Twoja odpowiedź:**

### 3.4. Znajdź lepszą prostą

Oblicz MSE prostej na danych uczących. Spróbuj dwóch lub trzech par `a`, `b` i porównaj wyniki.
Po zmianie parametrów przelicz `Y_hat`, zanim obliczysz błąd.

Na koniec zapisz najlepszą znalezioną parę jako `a_manual`, `b_manual`,
a jej MSE jako `mse_manual`. Czy udało się pobić przewidywanie średniej? Nie musisz znaleźć minimum.

<details>
<summary>Podpowiedź: kryterium oceny</summary>

Obliczasz MSE tak samo jak w części 2. Teraz odejmujesz predykcje prostej zamiast jednej stałej.

</details>

**Twoja odpowiedź:**

## 4. Co wybierze metoda najmniejszych kwadratów? (12 min)

### 4.1. Dopasuj prostą

Najpierw przewidź: czy procedura minimalizująca MSE może znaleźć prostą lepszą od Twojej?
Dopasuj regresję sprzedaży względem TV na próbie uczącej. Wynik zapisz jako `model_tv`.
Odczytaj współczynniki i porównaj je ze swoimi `a_manual`, `b_manual`.

Biblioteka nazywa wyraz wolny `Intercept`, a nachylenie `TV`. Odpowiadają one Twoim `a`, `b`
oraz współczynnikom $\widehateta_0$, $\widehateta_1$ z książki.

<details>
<summary>Podpowiedź 1: dopasowanie</summary>

`smf.ols("sales ~ TV", data=data_tr).fit()` dopasowuje model z wyrazem wolnym. Po lewej stronie `~` jest przewidywany wynik, po prawej cecha.

</details>

<details>
<summary>Podpowiedź 2: współczynniki</summary>

`model_tv.params` zawiera oszacowania współczynników.

</details>

**Twoja odpowiedź:**

### 4.2. Porównaj dopasowanie

Oblicz predykcje dopasowanego modelu na danych uczących jako `Y_hat_tr` oraz ich MSE jako `mse_tv_tr`.
Porównaj wyniki: średnia (`mse_mean`), Twoja prosta (`mse_manual`), prosta MNK (`mse_tv_tr`).
Czy wynik zgadza się z Twoim przewidywaniem?

Na tej samej próbie minimalizacja sumy kwadratów i średniej kwadratów wybiera tę samą prostą.

<details>
<summary>Podpowiedź: predykcje</summary>

`model_tv.predict(data_tr)` zwraca predykcję dla każdego rynku w podanej tabeli.

</details>

**Twoja odpowiedź:**

### 4.3. Jeden nowy budżet

Oblicz predykcję przy budżecie TV = 100 tys. USD ze współczynników modelu.
Następnie sprawdź ją metodą `predict`. Czy dostajesz ten sam wynik?

Zinterpretuj nachylenie: jaka różnica sprzedaży, w sztukach, odpowiada w tym modelu
budżetowi TV większemu o 1000 USD? Czy to dowód skutku zwiększenia wydatków?

<details>
<summary>Podpowiedź 1: wybrany współczynnik</summary>

`model_tv.params["TV"]` wybiera nachylenie. Wyraz wolny ma nazwę `Intercept`.

</details>

<details>
<summary>Podpowiedź 2: nowa obserwacja</summary>

`pd.DataFrame({"TV": [100.0]})` tworzy tabelę jednego rynku z kolumną wymaganą przez model.

</details>

**Twoja odpowiedź:**

## 5. Czy model pomaga na nowych danych? (15 min)

### 5.1. Oceń model TV

Czy prosta MNK musi pokonać przewidywanie średniej również na nowych rynkach?
Zapisz przewidywanie przed obliczeniem.

Otwórz próbę testową: oblicz predykcje dopasowanego już `model_tv` na `data_te`
i zapisz je jako `Y_hat_te`. Oblicz testowe MSE jako `mse_tv_te`.

<details>
<summary>Podpowiedź: nowa próba</summary>

W `predict` zmień tabelę na testową. Przy obliczaniu MSE użyj sprzedaży z tej samej tabeli. Nie dopasowuj modelu ponownie.

</details>

**Twoja odpowiedź:**

### 5.2. Porównaj z przewidywaniem średniej

Dla każdego rynku testowego przewiduj zapamiętaną wartość `mean_tr`.
Oblicz MSE tego modelu jako `mse_mean_te`. Nie licz nowej średniej na próbie testowej.

Wypisz wyniki obu modeli i odpowiedz: czy znajomość budżetu TV pomaga przewidywać sprzedaż
w tej próbie testowej? Czy przewaga na danych uczących sama w sobie gwarantowałaby taki wynik?

**Twoja odpowiedź:**

### 5.3. Jedna konkretna pomyłka

Odczytaj zaobserwowaną sprzedaż i predykcję modelu TV dla pierwszego rynku testowego.
Oblicz resztę. Podaj wszystkie trzy wartości w sztukach.
Dlaczego nawet dobra prosta nie musi trafiać w każdą obserwację?

<details>
<summary>Podpowiedź: jednostki</summary>

Pierwszy element wybierasz przez `.iloc[0]`. Przejście z tysięcy sztuk do sztuk wymaga pomnożenia przez 1000.

</details>

**Twoja odpowiedź:**

## 6. Samodzielnie: radio (15 min)

Czy samo radio przewiduje sprzedaż lepiej niż samo TV?

Zbuduj model przewidujący sprzedaż z budżetu radia. Użyj tych samych obserwacji uczących i testowych.
Porównaj go z modelem TV i przewidywaniem średniej. Napisz krótką odpowiedź z wynikiem liczbowym
oraz interpretacją nachylenia modelu radia w jednostkach danych.

Możesz korzystać ze swoich wcześniejszych komórek i notatek. Samodzielnie zdecyduj, jakie obliczenia wykonać.
W razie potrzeby dodawaj komórki.

Porównujemy dwa z góry wskazane modele na wspólnym teście. Nie dobieramy kolejnych wariantów modelu
według wyników testowych -- takie strojenie wymagałoby walidacji, jak na labie 3.

**Twoja odpowiedź:**


**Do wspólnego omówienia:** czy większa wartość nachylenia oznacza lepszy model?
Czy to porównanie wystarcza, żeby zalecić firmie przeniesienie budżetu między mediami?

## 7. Wnioski (5 min)

Bez nowych obliczeń odpowiedz własnymi słowami:

1. Co konkretnie zrobiła procedura dopasowania modelu?
2. Jak sprawdziłeś, czy model pomaga przewidywać?
3. Co mówi nachylenie, a czego nie dowodzi o skutkach zmiany budżetu?

**Twoje odpowiedzi:**

## Rozszerzenie: TV i radio razem (opcjonalnie)

Dopasuj model uwzględniający oba budżety na próbie uczącej.
Oblicz predykcje dla dwóch rynków: na obu budżet radia wynosi 20 tys. USD,
a budżet TV wynosi odpowiednio 100 i 110 tys. USD.

Porównaj predykcje. Jak ich różnica wiąże się ze współczynnikiem przy TV?
Co teraz oznacza „przy ustalonym budżecie radia”?


<details>
<summary>Podpowiedź: dwie cechy</summary>

W formule modelu kolejne cechy łączysz znakiem `+`. Nowa tabela do predykcji musi zawierać obie kolumny.

</details>

**Twoja odpowiedź:**

## Fragmenty książki

[Model liniowy](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-model-liniowy)
i [metoda najmniejszych kwadratów](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#sec-mnk)
wyjaśniają model i procedurę dopasowania.

Jeśli chcesz wyprowadzić wzory na współczynniki prostej, przejdź do
[zadania 6.1 a)](https://rodakt.github.io/MAD_book/2-modele-liniowe/06-regresja-liniowa.html#exr-mnk-regresja-prosta).